In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from tqdm import tqdm

In [7]:
class Config:
    BATCH_SIZE = 128
    LEARNING_RATE = 0.01
    EPOCHS = 30
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    NUM_CLASSES = 10

In [12]:
# 기본 Block
class ConvBNReLU(nn.Sequential):
    def __init__(self, in_planes, out_planes, kernel_size=3, stride=1, groups=1):
        padding = (kernel_size - 1) // 2
        super(ConvBNReLU, self).__init__(
            # Convolution
            nn.Conv2d(in_planes, out_planes, kernel_size, stride, padding, groups=groups, bias=False),

            # BN
            nn.BatchNorm2d(out_planes),

            # ReLU6
            nn.ReLU6(inplace=True) # MobileNet은 ReLU6를 사용
        )

# Inverted Residual Block
class InvertedResidual(nn.Module):
    def __init__(self, inp, oup, stride, expand_ratio):
        super(InvertedResidual, self).__init__()
        self.stride = stride

        # 확장 비율(Expantion Ratio) 계산
        hidden_dim = int(round(inp * expand_ratio))

        # Skip Connection은 덧셈연산이라 stride=1이고 입출력 채널 수가 같아야 함.
        self.use_res_connect = self.stride == 1 and inp == oup

        layers = []
        # expand_ratio가 1이면 궅이 해줄 필요 없으므로 생략
        if expand_ratio != 1:
            # Expand(Expressiveness)
            layers.append(ConvBNReLU(inp, hidden_dim, kernel_size=1))

        layers.extend([
            # Depthwise Conv: 공간 특징 추출
            # 입출력 채널 수를 동일하게 줄 때 Conv -> Depthwise Conv가 됨.
            ConvBNReLU(hidden_dim, hidden_dim, stride=stride, groups=hidden_dim),

            # Project: 다시 채널 줄이기 (Linear Bottleneck)
            # 여기선 ReLU 사용 X
            nn.Conv2d(hidden_dim, oup, 1, 1, 0, bias=False),
            nn.BatchNorm2d(oup),
        ])

        self.conv = nn.Sequential(*layers)

    def forward(self, x):
        if self.use_res_connect:
            return x + self.conv(x) # x + F(x) 잔차연결
        else:
            return self.conv(x)

In [13]:
class MobileNetV2(nn.Module):
    def __init__(self, num_classes=10):
        super(MobileNetV2, self).__init__()

        input_channel = 32
        last_channel = 1280

        # CIFAR-10 (32x32)데이터셋을 사용하여 원본보다 Stride를 줄여서 초반에 이미지 크기가 줄어드는 것을 방지함.
        inverted_residual_setting = [
            # t(확장), c(출력채널), n(반복), s(스트라이드)
            [1, 16, 1, 1],
            [6, 24, 2, 1], # stride 2 -> 1로 변경하여 크기 보존(CIFAR-10 에서만)
            [6, 32, 3, 2],
            [6, 64, 4, 2],
            [6, 96, 3, 1],
            [6, 160, 3, 2],
            [6, 320, 1, 1],
        ]

        # 첫 번째 Layer에서 Stride 2 -> 1로 변경
        self.features = [ConvBNReLU(3, input_channel, stride=1)]

        # Bottleneck Blocks
        for t, c, n, s in inverted_residual_setting:
            output_channel = c
            # 같은 블록을 반복할 때 첫 블록에서만 크기를 줄임.
            for i in range(n):
                stride = s if i == 0 else 1
                self.features.append(InvertedResidual(input_channel, output_channel, stride, expand_ratio=t))
                input_channel = output_channel

        # 마지막 Layer
        self.features.append(ConvBNReLU(input_channel, last_channel, kernel_size=1))
        self.features = nn.Sequential(*self.features)

        # 분류기 (Classifier)
        self.classifier = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(last_channel, num_classes),
        )

    def forward(self, x):
        x = self.features(x) # (Batch, 1280, 4, 4)
        x = x.mean([2, 3]) # Global Average Pooling (4x4를 압축시켜 1x1로 만듬)
        x = self.classifier(x)
        return x

In [14]:
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
train_loader = DataLoader(train_dataset, batch_size=Config.BATCH_SIZE, shuffle=True, num_workers=2)

test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)
test_loader = DataLoader(test_dataset, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=2)


In [11]:
model = MobileNetV2(num_classes=Config.NUM_CLASSES).to(Config.DEVICE)
criterion = nn.CrossEntropyLoss()
# SGD Optimizer + Momentum
optimizer = optim.SGD(model.parameters(), lr=Config.LEARNING_RATE, momentum=0.9, weight_decay=5e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=Config.EPOCHS)

def train(epoch):
    model.train()
    train_loss = 0
    correct = 0
    total = 0
    loop = tqdm(train_loader, desc=f"Epoch {epoch}")

    for inputs, targets in loop:
        inputs, targets = inputs.to(Config.DEVICE), targets.to(Config.DEVICE)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()

        loop.set_postfix(loss=train_loss/len(train_loader), acc=100.*correct/total)

def test(epoch):
    model.eval()
    test_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs, targets = inputs.to(Config.DEVICE), targets.to(Config.DEVICE)
            outputs = model(inputs)
            loss = criterion(outputs, targets)

            test_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

    print(f"Test | Loss: {test_loss/len(test_loader):.4f} | Acc: {100.*correct/total:.2f}%")

print(f"\n학습 시작 (Total Epochs: {Config.EPOCHS})")
for epoch in range(1, Config.EPOCHS + 1):
    train(epoch)
    test(epoch)
    scheduler.step()


학습 시작 (Total Epochs: 30)


Epoch 1: 100%|██████████| 391/391 [00:46<00:00,  8.45it/s, acc=35.7, loss=1.71]


Test | Loss: 1.3725 | Acc: 50.07%


Epoch 2: 100%|██████████| 391/391 [00:46<00:00,  8.44it/s, acc=55.1, loss=1.25]


Test | Loss: 1.0920 | Acc: 61.82%


...

Test | Loss: 0.3543 | Acc: 88.98%


Epoch 29: 100%|██████████| 391/391 [00:47<00:00,  8.23it/s, acc=94.7, loss=0.153]


Test | Loss: 0.3564 | Acc: 88.93%


Epoch 30: 100%|██████████| 391/391 [00:47<00:00,  8.23it/s, acc=94.8, loss=0.155]


Test | Loss: 0.3539 | Acc: 89.06%
